# Milestone 3: Knowledge State Extraction

This notebook runs learner-by-Knowledge-Component inference using the selected AKT checkpoint from Milestone 2. It does **not** train or fine-tune a model. It calls the same `export_states.py` implementation used locally so the notebook and script share validation and aggregation behavior.

The CSV, inference manifest, validation report, and a ZIP bundle are saved under `knowledge_tracing/artifacts/knowledge_states/` in Google Drive.

## 1. Mount Google Drive and select the project

The expected Drive folder is `MyDrive/thesis`, as in Milestones 1 and 2. This folder must contain the Milestone 1 maps and processed data, the Milestone 2 model configuration and trained checkpoint, and this repository's `knowledge_tracing` Python files.

In [1]:
import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/thesis').resolve()
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f'Project folder not found: {PROJECT_ROOT}. '
        'Update PROJECT_ROOT to the Drive folder containing data/ and knowledge_tracing/.'
    )
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'knowledge_tracing'))
print(f'Project root: {PROJECT_ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/thesis


## 2. Install and inspect the inference environment

The saved model was trained with `pykt-toolkit==0.0.38`. PyTorch is supplied by the Colab runtime; this cell does not reinstall it. A GPU is recommended, but CPU inference is supported.

In [2]:
%pip install -q pykt-toolkit==0.0.38

In [3]:
import importlib.metadata
import json

import sys
import types

try:
    import turtle  # pyKT may import this optional stdlib GUI module.
except Exception:
    turtle_stub = types.ModuleType('turtle')
    turtle_stub.forward = lambda *args, **kwargs: None
    sys.modules['turtle'] = turtle_stub

import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f"pykt-toolkit: {importlib.metadata.version('pykt-toolkit')}")
if importlib.metadata.version('pykt-toolkit') != '0.0.38':
    raise RuntimeError('This checkpoint expects pykt-toolkit==0.0.38.')

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
pykt-toolkit: 0.0.38


## 3. Verify Milestone 1 and 2 inputs

The selected configuration points to the trained AKT seed 43 checkpoint. If a required artifact is missing, copy or sync it from the previous milestone before proceeding.

In [4]:
CONFIG_DIR = PROJECT_ROOT / 'knowledge_tracing/configs'
selection_path = CONFIG_DIR / 'selected_model.json'
if not selection_path.is_file():
    raise FileNotFoundError(f'Selected model configuration not found: {selection_path}')
selection = json.loads(selection_path.read_text(encoding='utf-8'))
if selection.get('model_name') != 'akt':
    raise ValueError(f"This notebook supports AKT only; selected model is {selection.get('model_name')!r}.")

checkpoint_path = (PROJECT_ROOT / selection['checkpoint']).resolve()
required_files = [
    PROJECT_ROOT / 'data/raw/english_learning_events.csv',
    PROJECT_ROOT / 'data/processed/question_map.csv',
    PROJECT_ROOT / 'data/processed/concept_map.csv',
    PROJECT_ROOT / 'data/processed/split_assignments.csv',
    CONFIG_DIR / 'data_config.json',
    checkpoint_path,
    PROJECT_ROOT / 'knowledge_tracing/kt_common.py',
    PROJECT_ROOT / 'knowledge_tracing/kt_common.py',
    PROJECT_ROOT / 'knowledge_tracing/export_states.py',
    PROJECT_ROOT / 'knowledge_tracing/inference.py',
    PROJECT_ROOT / 'data/processed/pykt_english/test_sequences.csv',
    PROJECT_ROOT / 'knowledge_tracing/artifacts/evaluation/test_comparison.csv',
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('Required milestone artifacts are missing:\n' + '\n'.join(missing_files))

print(f"Selected model: {selection['model_name']} ({selection['model_version']})")
print(f"Checkpoint: {checkpoint_path}")
print(f"Validation AUC: {selection['validation_metrics']['auc']:.6f}")
print(f"Best validation epoch: {selection['best_epoch']}")
print('All source data, maps, configs, and checkpoint files are present.')

Selected model: akt (akt-seed43-best-validation-auc)
Checkpoint: /content/drive/MyDrive/thesis/knowledge_tracing/artifacts/akt/seed43/qid_model.ckpt
Validation AUC: 0.866965
Best validation epoch: 25
All source data, maps, configs, and checkpoint files are present.


## 4. Run learner-KC inference and export

This calls the repository's tested exporter. It loads the checkpoint, checks that the original ID maps match training, scores a deterministic sample of up to 10 questions per KC, aggregates question probabilities by mean, and validates the learner-KC table. No test split is used to train or select the model.

In [5]:
import pandas as pd

from knowledge_tracing.export_states import export_states

INPUT_PATH = PROJECT_ROOT / 'data/raw/english_learning_events.csv'
OUTPUT_DIR = PROJECT_ROOT / 'knowledge_tracing/artifacts/knowledge_states'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

states = export_states(
    input_path=INPUT_PATH,
    output_dir=OUTPUT_DIR,
    max_questions_per_kc=10,
    batch_size=128,
    device=DEVICE,
)
print(f'Inference device: {DEVICE}')

[knowledge-state] Loading answer events from /content/drive/MyDrive/thesis/data/raw/english_learning_events.csv
[knowledge-state] Loaded 75,006 answer events (no as_of cutoff)
[knowledge-state] Validated maps: 2,189 questions, 96 KCs, 1,200 assigned learners
[knowledge-state] Loading selected AKT checkpoint; scoring all 2,189 mapped questions per learner
[knowledge-state] Scoring 1,200 learners in batches of 128 on cuda
[knowledge-state] Scored 60/1,200 learners (5%)
[knowledge-state] Scored 120/1,200 learners (10%)
[knowledge-state] Scored 180/1,200 learners (15%)
[knowledge-state] Scored 240/1,200 learners (20%)
[knowledge-state] Scored 300/1,200 learners (25%)
[knowledge-state] Scored 360/1,200 learners (30%)
[knowledge-state] Scored 420/1,200 learners (35%)
[knowledge-state] Scored 480/1,200 learners (40%)
[knowledge-state] Scored 540/1,200 learners (45%)
[knowledge-state] Scored 600/1,200 learners (50%)
[knowledge-state] Scored 660/1,200 learners (55%)
[knowledge-state] Scored 720

In [ ]:
# Milestone 2 reproduction check: does this pipeline reproduce the saved test metrics?
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

from inference import load_selected_model, score_akt_next_interactions

test = pd.read_csv(PROJECT_ROOT / 'data/processed/pykt_english/test_sequences.csv')
concept_map = pd.read_csv(PROJECT_ROOT / 'data/processed/concept_map.csv')
dimension_of = dict(zip(concept_map['concepts'], concept_map['kc_dimension']))

# One prediction target per answer except each learner's first (same targets as pyKT evaluation).
hist_q, hist_c, hist_r, tgt_q, tgt_c, truth = [], [], [], [], [], []
for _, row in test.iterrows():
    q = [int(v) for v in row['questions'].split(',')]
    c = [int(v) for v in row['concepts'].split(',')]
    r = [int(v) for v in row['responses'].split(',')]
    n = sum(int(v) != -1 for v in row['selectmasks'].split(','))   # drop padding
    for t in range(1, n):
        hist_q.append(q[:t]); hist_c.append(c[:t]); hist_r.append(r[:t])
        tgt_q.append(q[t]); tgt_c.append(c[t]); truth.append(r[t])
truth = np.asarray(truth)

model, device, _, _ = load_selected_model(DEVICE)
scores = score_akt_next_interactions(
    model, device, hist_c, hist_r, hist_q, tgt_q, tgt_c, maxlen=200, batch_size=256
)

p = np.clip(scores, 1e-7, 1 - 1e-7)
measured = {
    'auc': float(roc_auc_score(truth, p)),
    'accuracy': float(accuracy_score(truth, p >= 0.5)),
    'log_loss': float(log_loss(truth, p, labels=[0, 1])),
    'brier_score': float(np.mean((p - truth) ** 2)),
}
saved = selection['test_metrics']
rows = [(k, saved[k], measured[k], abs(saved[k] - measured[k])) for k in measured]
report = pd.DataFrame(rows, columns=['metric', 'saved (Milestone 2)', 'reproduced here', 'abs diff'])
display(report.round(5))
print(f'Targets scored: {len(truth):,}  (Milestone 2 reported 14,723)')

by_dim = pd.DataFrame({'dim': [dimension_of[c] for c in tgt_c], 'y': truth, 'p': scores})
display(by_dim.groupby('dim').apply(lambda g: pd.Series({
    'targets': len(g), 'auc': roc_auc_score(g['y'], g['p']),
    'accuracy': accuracy_score(g['y'], g['p'] >= 0.5)})).round(4))

TOL = 1e-3
passed = bool((report['abs diff'] < TOL).all())
(OUTPUT_DIR / 'model_fidelity_check.json').write_text(json.dumps(
    {'passed': passed, 'tolerance': TOL, 'n_targets': int(len(truth)), 'measured': measured, 'saved': saved}, indent=2))
if not passed:
    raise RuntimeError('Pipeline does NOT reproduce the Milestone 2 test metrics - do not use the exported states.')
print('PASS: the extraction pipeline reproduces the saved Milestone 2 test metrics.')

## 5. Review exported states and validation results

`predicted_correctness` estimates correctness on representative questions. It is not a probability of mastery. `observed_accuracy` is calculated from the learner's direct historical answers to that KC; it remains null for cold-start states.

In [6]:
from zipfile import ZIP_DEFLATED, ZipFile

manifest_path = OUTPUT_DIR / 'inference_manifest.json'
report_path = OUTPUT_DIR / 'knowledge_state_validation_report.md'
csv_path = OUTPUT_DIR / 'learner_skill_state.csv'
zip_path = OUTPUT_DIR / 'milestone3_knowledge_state_outputs.zip'
for path in (csv_path, manifest_path, report_path):
    if not path.is_file():
        raise FileNotFoundError(f'Expected export was not written: {path}')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest.get('validation_status') != 'pass':
    raise RuntimeError(f"Export validation did not pass: {manifest.get('validation_status')}")

with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in (csv_path, manifest_path, report_path):
        archive.write(path, arcname=path.name)
with ZipFile(zip_path) as archive:
    if archive.testzip() is not None:
        raise RuntimeError(f'Corrupt export archive: {zip_path}')

summary = states.groupby('state_status').size().rename('rows')
print(f"Learners: {states['learner_id'].nunique():,}")
print(f"Knowledge Components: {states['kc_id'].nunique():,}")
print(f"Learner-KC rows: {len(states):,}")
display(summary.to_frame())

dimension_summary = states.groupby('kc_dimension').agg(
    learner_kc_rows=('predicted_correctness', 'size'),
    mean_predicted_correctness=('predicted_correctness', 'mean'),
    mean_observed_accuracy=('observed_accuracy', 'mean'),
    mean_attempt_count=('attempt_count', 'mean'),
).round(4)
display(dimension_summary)

print(f"Checkpoint SHA-256: {manifest['checkpoint_sha256']}")
print(f"Training versions: {manifest['training_environment']}")
print(f"Runtime versions: {manifest['runtime_environment']}")
print(f'Validation status: {manifest["validation_status"].upper()}')

Learners: 1,200
Knowledge Components: 96
Learner-KC rows: 115,200


,rows
state_status,
cold_start,79289
estimated,35911


,learner_kc_rows,mean_predicted_correctness,mean_observed_accuracy,mean_attempt_count
kc_dimension,,,,
grammar,33600,0.7216,0.6829,0.6694
reading,28800,0.6631,0.6290,0.6192
vocabulary,31200,0.7210,0.6879,0.7287
writing,21600,0.6669,0.6393,0.5530


Checkpoint SHA-256: e2b7c46f7ba548ac71e344bd9c0e9771a90d2a41d824c95ea1eb084101dd9a9a
Training versions: {'pykt_toolkit_version': '0.0.38', 'torch_version': '2.14.1+cu130'}
Runtime versions: {'pykt_toolkit_version': '0.0.38', 'torch_version': '2.14.1+cu130'}
Validation status: PASS


In [7]:
from IPython.display import Markdown, display

display(Markdown(report_path.read_text(encoding='utf-8')))
print('\nSaved files on Google Drive:')
for path in (csv_path, manifest_path, report_path, zip_path):
    print(f'- {path} ({path.stat().st_size:,} bytes)')

# Knowledge-State Extraction Validation

- Validation status: PASS
- Learners: 1,200
- Knowledge Components: 96
- Learner-KC records: 115,200
- Learner-KC coverage: 100.0%
- Source answer events: 75,006
- Snapshot cutoff (UTC): not specified
- State status counts: {'cold_start': 79289, 'estimated': 35911}
- Evidence tier counts: {'none': 79289, 'low': 25932, 'moderate': 8692, 'high': 1287}
- Estimated states with one direct KC attempt: 16,855 (46.9%)
- Predictions at or beyond 0.05/0.95: 30,394
- Predicted correctness is an estimated probability of answering a mapped question correctly; it is not a mastery probability.

## Structural Checks

- All structural checks passed.

## Distributions

Attempt count summary:

```text
count    115200.000000
mean          0.651094
std           1.262733
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          14.000000
```

Predicted correctness summary:

```text
count    115200.000000
mean          0.696564
std           0.295297
min           0.014922
25%           0.477192
50%           0.816728
75%           0.949658
max           0.998599
```

Observed accuracy summary (direct evidence only):

```text
count    35911.00000
mean         0.66329
std          0.43071
min          0.00000
25%          0.00000
50%          1.00000
75%          1.00000
max          1.00000
```

Prediction minus observed historical KC accuracy, descriptive comparison only:

```text
count    35911.000000
mean         0.044327
std          0.287965
min         -0.970141
25%         -0.091875
50%         -0.019851
75%          0.171691
max          0.995341
```

## Split-Level Descriptive Comparison

The following compares full-map model estimates with each learner's historical KC accuracy through the cutoff. It is not a calibration/error estimate: train learners contributed to model fitting, and the compared quantities average over different questions.

| Split | Learners | Learner-KC states | Evidence-backed learner-KCs | Mean predicted | Mean observed | Mean predicted minus observed | Single-attempt learner-KCs |
|---|---:|---:|---:|---:|---:|---:|---:|
| test | 240 | 23040 | 7057 | 0.6979 | 0.6524 | 0.0455 | 3288 |
| train | 840 | 80640 | 25114 | 0.7144 | 0.6716 | 0.0429 | 11734 |
| valid | 120 | 11520 | 3740 | 0.6801 | 0.6284 | 0.0517 | 1833 |

## Dimension Summary

| Dimension | States | Mean predicted | Predicted std. dev. | Mean observed | Mean prediction minus observed |
|---|---:|---:|---:|---:|---:|
| grammar | 33600 | 0.7216 | 0.2769 | 0.6829 | 0.0472 |
| reading | 28800 | 0.6631 | 0.2986 | 0.6290 | 0.0494 |
| vocabulary | 31200 | 0.7210 | 0.2805 | 0.6879 | 0.0399 |
| writing | 21600 | 0.6669 | 0.3303 | 0.6393 | 0.0396 |

## All-Question vs 10-Question Sensitivity

- Mean absolute difference across all learner-KCs: 0.0014
- 95th percentile absolute difference: 0.0053
- Primary predictions average over every mapped question in the KC. The deterministic 10-question estimate is retained separately as a sensitivity check from the same model scores.

## Interpretation and Limitations

- Historical observed accuracy is reported separately from the model prediction.
- Cold-start rows have no direct KC attempts and a null observed accuracy; their predicted correctness can still use the learner's history in other KCs, or the model's learned prior when history is empty.
- Evidence tiers are none (0 attempts), low (1-2), moderate (3-5), and high (6 or more); consumers should treat low-evidence estimates cautiously.
- The checkpoint was selected by validation AUC only. Held-out test metrics are descriptive and were not used for model selection.



Saved files on Google Drive:
- /content/drive/MyDrive/thesis/knowledge_tracing/artifacts/knowledge_states/learner_skill_state.csv (35,658,453 bytes)
- /content/drive/MyDrive/thesis/knowledge_tracing/artifacts/knowledge_states/inference_manifest.json (4,762 bytes)
- /content/drive/MyDrive/thesis/knowledge_tracing/artifacts/knowledge_states/knowledge_state_validation_report.md (3,674 bytes)
- /content/drive/MyDrive/thesis/knowledge_tracing/artifacts/knowledge_states/milestone3_knowledge_state_outputs.zip (3,531,657 bytes)


## Outputs and interpretation

The validated artifacts are saved to `MyDrive/thesis/knowledge_tracing/artifacts/knowledge_states/`:

- `learner_skill_state.csv` — one learner-KC record, including historical evidence, estimated correctness, and model provenance.
- `inference_manifest.json` — input, mapping, checkpoint, runtime, and execution metadata.
- `knowledge_state_validation_report.md` — structural checks, distributions, and dimension summaries.
- `milestone3_knowledge_state_outputs.zip` — downloadable bundle of the three outputs above.

`predicted_correctness` is not mastery. Cold-start predictions can use other KC history or the trained model prior, and are explicitly marked `cold_start` with no fabricated observed accuracy. Re-running this notebook overwrites these exports with a new inference timestamp.

In [9]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/thesis/knowledge_tracing/artifacts/knowledge_states/learner_skill_state.csv"
# Load the dataset
df = pd.read_csv(CSV_PATH)

# Check the shape (number of rows and columns)
print("Dataset Shape:", df.shape)

# Check the column names
print("\nColumns:")
print(df.columns.tolist())

# View the first 5 rows
display(df.head())

Dataset Shape: (115200, 21)

Columns:
['learner_id', 'split', 'kc_id', 'kc_dimension', 'attempt_count', 'observed_accuracy', 'predicted_correctness', 'predicted_correctness_10q', 'last_attempt_at', 'model_name', 'model_version', 'checkpoint_id', 'checkpoint_sha256', 'inference_timestamp', 'as_of', 'evidence_count', 'evidence_tier', 'prediction_question_count', 'sensitivity_question_count', 'aggregation_method', 'state_status']


,learner_id,split,kc_id,kc_dimension,attempt_count,observed_accuracy,predicted_correctness,predicted_correctness_10q,last_attempt_at,model_name,...,checkpoint_id,checkpoint_sha256,inference_timestamp,as_of,evidence_count,evidence_tier,prediction_question_count,sensitivity_question_count,aggregation_method,state_status
0,L0001,train,KC001,grammar,5,0.8,0.967736,0.968169,2023-10-14T09:30:14+00:00,akt,...,akt-seed43-qid-model,e2b7c46f7ba548ac71e344bd9c0e9771a90d2a41d824c9...,2026-11-08T08:40:57.441491+00:00,NaN,5,moderate,25,10,arithmetic_mean_of_all_mapped_question_probabi...,estimated
1,L0001,train,KC002,grammar,0,NaN,0.736973,0.732547,NaN,akt,...,akt-seed43-qid-model,e2b7c46f7ba548ac71e344bd9c0e9771a90d2a41d824c9...,2026-11-08T08:40:57.441491+00:00,NaN,0,none,27,10,arithmetic_mean_of_all_mapped_question_probabi...,cold_start
2,L0001,train,KC003,grammar,3,1.0,0.994856,0.994912,2023-10-19T08:13:03+00:00,akt,...,akt-seed43-qid-model,e2b7c46f7ba548ac71e344bd9c0e9771a90d2a41d824c9...,2026-11-08T08:40:57.441491+00:00,NaN,3,moderate,28,10,arithmetic_mean_of_all_mapped_question_probabi...,estimated
3,L0001,train,KC004,grammar,0,NaN,0.185262,0.186629,NaN,akt,...,akt-seed43-qid-model,e2b7c46f7ba548ac71e344bd9c0e9771a90d2a41d824c9...,2026-11-08T08:40:57.441491+00:00,NaN,0,none,26,10,arithmetic_mean_of_all_mapped_question_probabi...,cold_start
4,L0001,train,KC005,grammar,0,NaN,0.580557,0.576529,NaN,akt,...,akt-seed43-qid-model,e2b7c46f7ba548ac71e344bd9c0e9771a90d2a41d824c9...,2026-11-08T08:40:57.441491+00:00,NaN,0,none,27,10,arithmetic_mean_of_all_mapped_question_probabi...,cold_start


In [10]:
# Disconnect runtime to save GPU usage
from google.colab import runtime
runtime.unassign()